# Kararı açıkla, kaydı düzelt
**Bölüm 33–34 · Yalçın Kavaklı ve Murat Şahin**

Amaç: Aynı modelin verdiği kararın gerekçesini okuyup hatalı veri girişini düzelttiğimizde ne değiştiğini görmek.

33. bölümdeki *Trust Fall* sorgulanamayan bir karar sistemini anlatır. Biz kredi verisi üzerinden açıklama, kayıt doğrulama ve yeniden değerlendirme adımlarını canlandırıyoruz. Bu gerçek bir başvuru, gerçek banka politikası veya kitabın orijinal kodu değildir.

**Sıra:** Hücreleri yukarıdan aşağı çalıştır. Veri dosyası yoksa UCI’dan indirilir; Colab’da internet gerekir. Bu defterin hücreleri yerel Python’da çalıştırılmıştır; yeni defter Colab arayüzünde ayrıca denenmelidir.


## 1. Kütüphaneler ve değişken adları
Eksik paket hatası alırsan ayrı hücrede `%pip install numpy scikit-learn` çalıştır.

In [1]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
from io import BytesIO
import hashlib
import json
import numpy as np
import sklearn
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

SOURCE_URL = 'https://archive.ics.uci.edu/static/public/573/south%2Bgerman%2Bcredit%2Bupdate.zip'
FEATURES = ['Vade (ay)', 'Tutar (dönüştürülmüş tarihsel değer)',
            'Hesap yok', 'Hesap: 0 DM altı', 'Hesap: 0–200 DM', 'Hesap: en az 200 DM / maaş']
ACCOUNT_LABELS = ['Hesap yok', '0 DM altı', '0–200 DM arası', 'En az 200 DM / en az 1 yıllık maaş aktarımı']



## 2. Gerçek veriyi yükle
South German Credit: 1.000 tarihsel kredi kaydı, 1973–1975. Kaynakta 20 girdi var; öğretim için **vade, tarihsel tutar ve hesap durumu** seçildi. Hesap kategorileri sayısal büyüklük değil, ayrı göstergelerdir.

Kaynak hedef `kredit`: 0=olumsuz, 1=olumlu. Kodda olumsuz sınıfı 1 yapmak için `1 - kredit` kullanılır. Hedef girdilerde kullanılmaz.

Veri dosyası varsa internet bağlantısı gerekmez.


In [2]:
def load_data(folder=Path('.')):
    folder=Path(folder)
    file=folder/'SouthGermanCredit.asc'
    if not file.exists():
        with urlopen(SOURCE_URL,timeout=60) as response:
            archive=ZipFile(BytesIO(response.read()))
        for name in ['SouthGermanCredit.asc','codetable.txt','read_SouthGermanCredit.R']:
            if name in archive.namelist():
                (folder/name).write_bytes(archive.read(name))
    data=np.loadtxt(file,skiprows=1,dtype=int)
    assert data.shape==(1000,21)
    # Model yalnızca üç gerçek kavramı kullanır. Kategoriler sıralı sayı sayılmaz.
    X=np.column_stack([data[:,1],data[:,4]]+[(data[:,0]==v).astype(int) for v in range(1,5)])
    # Kaynak hedef: 0=bad, 1=good. Burada 1=olumsuz sonuç olarak dönüştürülür.
    y=1-data[:,-1]
    return data,X,y


data,X,y=load_data()
print("Ham veri:",data.shape,"; model girdisi:",X.shape)
print("Olumlu:",int((y==0).sum()),"Olumsuz:",int((y==1).sum()))
print("2. kaydın vadesi:",int(data[1,1]),"ay")

Ham veri: (1000, 21) ; model girdisi: (1000, 6)
Olumlu: 700 Olumsuz: 300
2. kaydın vadesi: 9 ay


## 3. Açıklama ve değerlendirme yardımcıları
`explain`, ağacın gerçekten izlediği koşulları okur. Modelin yerine başka bir açıklama modeli kurulmaz. Bu, küçük bir modelin kendi içinden açıklanmasıdır; SHAP veya LIME değildir.

`evaluate`, seçtiğimiz eşikle 300 test kaydındaki dört sonucu sayar.


In [3]:
def encode(duration,amount,account):
    return np.array([duration,amount]+[int(account==v) for v in range(1,5)],dtype=float)

def explain(model,row):
    """Eğitilmiş ağacın gerçekten izlediği kurallar; yaklaşık açıklama değil."""
    tree=model.tree_
    node=0
    rules=[]
    while tree.children_left[node]!=tree.children_right[node]:
        feature=int(tree.feature[node]); threshold=float(tree.threshold[node])
        go_left=bool(row[feature]<=threshold)
        if feature<2:
            text=f'{FEATURES[feature]}: {row[feature]:g} '+('≤' if go_left else '>')+f' {threshold:g}'
        else:
            text=f'{FEATURES[feature]}: '+('hayır' if go_left else 'evet')
        rules.append({'node':node,'feature':feature,'threshold':threshold,'left':go_left,'text':text})
        node=int(tree.children_left[node] if go_left else tree.children_right[node])
    score=float(model.predict_proba([row])[0,1])
    return {'rules':rules,'leaf':node,'score':score,'leaf_train_count':int(tree.n_node_samples[node])}

def evaluate(y,score,threshold=.5):
    prediction=np.asarray(score)>=threshold
    tn,fp,fn,tp=confusion_matrix(y,prediction,labels=[0,1]).ravel()
    return {'threshold':threshold,'accuracy':float(accuracy_score(y,prediction)),
            'balanced_accuracy':float(balanced_accuracy_score(y,prediction)),
            'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}



## 4. Eğit ve canlandırmayı çalıştır
700 eğitim ve 300 test kaydı. Sınıf oranları korunur, tohum 42’dir. Karar ağacı en fazla 3 seviyeli, yaprakta en az 30 örneklidir.

**Canlandırma:** 2. gerçek kaydın vadesi 9 aydır. Önce vade sanki 48 ay girilmiş gibi tahmin üretir, sonra kaynak değeri geri veririz. Yanlış vade olayını biz ekledik; bu veride belgelenmiş bir itiraz değildir. Örnek, farkı anlatabilmek için seçildi; tüm başvurularda aynı etki beklenmez.

Örnek ekran politikası: yapraktaki olumsuz oranı ≥ %50 ise “Ek inceleme”; daha düşükse “Ön değerlendirmeyi geçer”. Bu politika gerçek bankanın politikası değildir.


In [4]:
def run(folder=Path('.')):
    folder=Path(folder)
    data,X,y=load_data(folder)
    train,test=train_test_split(np.arange(len(y)),test_size=.3,stratify=y,random_state=42)
    model=DecisionTreeClassifier(max_depth=3,min_samples_leaf=30,random_state=42)
    model.fit(X[train],y[train])
    scores=model.predict_proba(X[test])[:,1]
    # Kaynak veri satırı 2 (başlık hariç); test kümesindedir. Kimlik bilgisi yoktur.
    # Sunum için sonuç değiştiren bir örnek seçildi; test başarısı bu seçimle ayarlanmadı.
    i=1
    assert i in test and data[i,1]==9
    correct=X[i].copy();wrong=correct.copy();wrong[0]=48
    assert model.predict_proba([wrong])[0,1]>=.5>model.predict_proba([correct])[0,1]
    t=model.tree_
    nodes=[]
    for n in range(t.node_count):
        nodes.append({'left':int(t.children_left[n]),'right':int(t.children_right[n]),
            'feature':int(t.feature[n]),'threshold':float(t.threshold[n]),
            'score':float(t.value[n][0,1]/t.value[n][0].sum()),'count':int(t.n_node_samples[n])})
    # Tarayıcıda çalışan aynı ağacın sonucu Python ile bütün gerçek satırlarda eşleşmeli.
    for row,expected in zip(X,model.predict_proba(X)[:,1]):
        n=0
        while nodes[n]['left']!=-1:
            q=nodes[n];n=q['left'] if row[q['feature']]<=q['threshold'] else q['right']
        assert abs(nodes[n]['score']-float(expected))<1e-12
    result={'data':{'name':'South German Credit','source':SOURCE_URL,'rows':1000,
        'predictor_concepts':3,'raw_predictors':20,'train':700,'test':300,
        'sha256':hashlib.sha256((folder/'SouthGermanCredit.asc').read_bytes()).hexdigest()},
        'versions':{'numpy':np.__version__,'scikit-learn':sklearn.__version__},
        'features':FEATURES,'account_labels':ACCOUNT_LABELS,'nodes':nodes,
        'metrics':evaluate(y[test],scores),
        'test':{'labels':y[test].tolist(),'scores':scores.tolist()},
        'case':{'source_row':2,'source_label':int(y[i]),'account':int(data[i,0]),
            'amount':int(data[i,4]),'correct_duration':9,'wrong_duration':48,
            'correct':explain(model,correct),'wrong':explain(model,wrong)},
        'tree_text':export_text(model,feature_names=FEATURES)}
    (folder/'sonuclar.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
    print('700 eğitim / 300 test; üç gerçek kavram; derinlik en fazla 3.')
    print('Test sonuçları:',result['metrics'])
    for name,row in [('Canlandırma: yanlış vade 48 ay',wrong),('Kaynak kayda dönüş: 9 ay',correct)]:
        explanation=explain(model,row)
        print('\n'+name)
        for rule in explanation['rules']:print(' →',rule['text'])
        print('Yaprak olumsuz sonuç oranı:',round(explanation['score'],4))
        print('Örnek politika:', 'Ek inceleme' if explanation['score']>=.5 else 'Ön değerlendirmeyi geçer')
    print('\n',result['tree_text'])
    return model,result


model,result=run()

700 eğitim / 300 test; üç gerçek kavram; derinlik en fazla 3.
Test sonuçları: {'threshold': 0.5, 'accuracy': 0.7066666666666667, 'balanced_accuracy': 0.6190476190476191, 'tn': 176, 'fp': 34, 'fn': 54, 'tp': 36}

Canlandırma: yanlış vade 48 ay
 → Hesap: en az 200 DM / maaş: hayır
 → Vade (ay): 48 > 11.5
 → Hesap yok: evet
Yaprak olumsuz sonuç oranı: 0.5576
Örnek politika: Ek inceleme

Kaynak kayda dönüş: 9 ay
 → Hesap: en az 200 DM / maaş: hayır
 → Vade (ay): 9 ≤ 11.5
 → Hesap: 0 DM altı: hayır
Yaprak olumsuz sonuç oranı: 0.1429
Örnek politika: Ön değerlendirmeyi geçer

 |--- Hesap: en az 200 DM / maaş <= 0.50
|   |--- Vade (ay) <= 11.50
|   |   |--- Hesap: 0 DM altı <= 0.50
|   |   |   |--- class: 0
|   |   |--- Hesap: 0 DM altı >  0.50
|   |   |   |--- class: 0
|   |--- Vade (ay) >  11.50
|   |   |--- Hesap yok <= 0.50
|   |   |   |--- class: 0
|   |   |--- Hesap yok >  0.50
|   |   |   |--- class: 1
|--- Hesap: en az 200 DM / maaş >  0.50
|   |--- Tutar (dönüştürülmüş tarihsel değer)

## 5. Kendi değişikliğini yap
`duration` değerini önce 48, sonra 9 yap. Diğer iki değeri sabit tut. Model yeniden eğitilmeden tahmin ve kural yolu değişir.

Burada basılan skor, eğitim yaprağındaki olumsuz kayıt oranıdır; güncel, kalibre edilmiş bireysel risk olasılığı değildir.


In [5]:
duration=48  # Sonra 9 yapıp yalnızca bu hücreyi yeniden çalıştır.
row=encode(duration,result['case']['amount'],result['case']['account'])
answer=explain(model,row)
print('Vade:',duration,'ay')
for rule in answer['rules']: print(rule['text'])
print('Yaprak olumsuz oranı:',round(100*answer['score'],1),'%')
print('Örnek karar:', 'Ek inceleme' if answer['score']>=.5 else 'Ön değerlendirmeyi geçer')

Vade: 48 ay
Hesap: en az 200 DM / maaş: hayır
Vade (ay): 48 > 11.5
Hesap yok: evet
Yaprak olumsuz oranı: 55.8 %
Örnek karar: Ek inceleme


## 6. Eşiği değiştir, hata sayılarını karşılaştır
Aynı modele ve aynı 300 test kaydına farklı karar eşikleri uygula. **Yanlış alarm:** gerçekte olumlu krediyi ek incelemeye yönlendirme. **Kaçırma:** gerçekte olumsuz krediyi ön değerlendirmeden geçirme.

Eşiği test sonuçlarına bakıp seçersen, aynı test kümesindeki başarı tarafsız bir son değerlendirme olmaz. Bu ekran eğitim içindir; gerçek model/politika seçiminde ayrı doğrulama verisi gerekir.


In [6]:
for threshold in [.3,.5,.6,.7]:
    metrics=evaluate(result['test']['labels'],result['test']['scores'],threshold)
    print('Eşik',threshold,'| yanlış alarm',metrics['fp'],'| kaçırma',metrics['fn'],
          '| doğruluk',round(metrics['accuracy'],3),'| dengeli doğruluk',round(metrics['balanced_accuracy'],3))

Eşik 0.3 | yanlış alarm 90 | kaçırma 18 | doğruluk 0.64 | dengeli doğruluk 0.686
Eşik 0.5 | yanlış alarm 34 | kaçırma 54 | doğruluk 0.707 | dengeli doğruluk 0.619
Eşik 0.6 | yanlış alarm 0 | kaçırma 90 | doğruluk 0.7 | dengeli doğruluk 0.5
Eşik 0.7 | yanlış alarm 0 | kaçırma 90 | doğruluk 0.7 | dengeli doğruluk 0.5


## Sonuçları sınıfta nasıl savunacağım?
- **Açıklama ne sağladı?** Sistemin hangi kayda ve kurala dayandığını gördük; yanlış vade girişini kontrol edebildik.
- **XAI hatayı kendi başına düzeltti mi?** Hayır. Kaynağı biz doğrulayıp girdiyi düzelttik, aynı modelle tekrar değerlendirdik.
- **Sistem artık kusursuz mu?** Hayır. Test hataları var; doğru girdiyle de yanlış karar verebilir.
- **Neden eşik var?** Modelin verdiği skorla, bu skora hangi işlemin uygulanacağı ayrı kararlardır. Hedefi doğru tanımlama tartışmasına bağlanır.
- **Veri günümüzü temsil ediyor mu?** Hayır. Tarihsel örnek ve olumsuz krediler fazla örneklenmiş; oranlar gerçek başvuru riskine aktarılamaz.
- **Kredi adaleti kanıtlandı mı?** Hayır. Bazı kişisel alanları kullanmamak, dolaylı ayrımcılığı ortadan kaldırdığımızı göstermez.

Kaynaklar: [UCI South German Credit](https://archive.ics.uci.edu/dataset/573/south%2Bgerman%2Bcredit%2B), [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html), Molnar [33](https://christophm.github.io/interpretable-ml-book/storytime.html) ve [34](https://christophm.github.io/interpretable-ml-book/future.html).
